In [1]:
import spectfbcalc_lib as sfc
from matplotlib import pyplot as plt
import xarray as xr
import yaml
import glob
from climtools import climtools_lib as ctl
import numpy as np
import os
from scipy import stats
import output_lib as out

No DISPLAY variable set. Switching to agg backend


In [2]:
#CARICARE DATI 
config_file = '/perm/ecme5989/SpectFbCalc/spectfbcalc/config_r8.yaml'
control, experiment, kernel = sfc.preprocess_data(config_file)
cart_out='/perm/ecme5989/new_func_prove/model_wNET/remapped_HUANG/'
fb=sfc.calc_fb(experiment, control, kernel, cart_out)
fb_inter=sfc.calc_fb_interannual(experiment, control, kernel, cart_out)

Time range for climatology: {'start': '2540-01-16 12:00:00', 'end': '2689-12-16 12:00:00'}
Time range for experiment: {'start': '1850-01-16 12:00:00', 'end': '1999-12-16 12:00:00'}
Loading kernel: HUANG

 -------> Loading control
PI already remapped
Computing albedo from rsus and rsds
Applying log to hus
Creating Net TOA variables
tas loaded
alb loaded
ta loaded
rlutcs loaded
rsutcs loaded
rsdt loaded
ts loaded
rsut loaded
rlut loaded
hus_log loaded
check vertical dimension
Rewriting vertical coordinates from Pa to hPa

 -------> Loading experiment
4x already remapped
Computing albedo from rsus and rsds
Applying log to hus
Creating Net TOA variables
tas loaded
alb loaded
ta loaded
rlutcs loaded
rsutcs loaded
rsdt loaded
ts loaded
rsut loaded
rlut loaded
hus_log loaded
check vertical dimension
Rewriting vertical coordinates from Pa to hPa

 -------> Computing climatology and anomalies


/perm/ecme5989/miniforge3/envs/spectfbcalc/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: divide by zero encountered in divide
  return self.func(*new_argspec)



 -------> Recomputing atm dp with surface pressure

Loading surface pressure data...


/etc/ecmwf/nfs/dh1_10_perm_b/ecme5989/SpectFbCalc/spectfbcalc/spectfbcalc_lib.py:320: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  surf_pressure = xr.open_mfdataset(ps_files, combine='by_coords', decode_times=time_coder)


Regridding...
Regridding completed in 0:00:07.538773
Surface pressure not available in experiment 4x, using default dp for kernel HUANG

 ----------> Preprocessing complete for model_wNET <------------

planck surf
albedo
planck atm
w-v
cloud
feedback calculation...
planck surf
albedo
planck atm
w-v
cloud
feedback calculation...


In [3]:
def somma(dRt, sky='clr'):
  
    if sky == 'clr':
        # prendi solo le componenti clear-sky
        arrays = [v for (s, comp), v in dRt.items() if s == 'clr']
    else:
        # prendi tutte le componenti all-sky + cloud
        arrays = [
            v for (s, comp), v in dRt.items()
            if s != 'clr' or comp == 'cloud'
        ]

    # somma tutte le DataArray
    dRt_sum = sum(arrays[1:], arrays[0])
    
    return dRt_sum
cart='/perm/ecme5989/new_func_prove/model_wNET/remapped_HUANG/no_mask/'
dRt=sfc.open_dRt(cart)
default_labels = ["Albedo", "Lapse Rate", "Planck Atmos", "Planck Surface", "Water Vapor", 'Cloud']
dRt_sum_cloud = somma(dRt, sky='cld')
dRt_sum = somma(dRt, sky='clr')

In [4]:
NET_clr=ctl.global_mean(experiment.ds_anom['Net0'].groupby("time.year").mean(dim='time'))
NET_cld=ctl.global_mean(experiment.ds_anom['Net'].groupby("time.year").mean(dim='time'))

In [ ]:
#TEST SU WV ERA5
path='/perm/ecme5989/new_func_prove/model_1/remapped_HUANG/dRt_water-vapor_global_cld.nc'
new=xr.open_dataset(path)
path='/perm/ecme5989/new_func_prove/provaERA_log/model_1/remapped_ERA5/dRt_water-vapor_global_cld.nc'
old=xr.open_dataset(path)
plt.figure()
new['__xarray_dataarray_variable__'].plot( label='HUANG')
old['__xarray_dataarray_variable__'].plot( label='ERA5_log')
plt.legend()
#plt.title('Radiation anomalies lapse-rate')
plt.savefig('/perm/ecme5989/new_func_prove/provaERA_log/model_1/remapped_ERA5/water-vapor_test.pdf')

In [5]:
##PLOT ZELINKA HUANG
ac = (fb['fb_coeffs'][('cld', 'planck-surf')].slope + fb['fb_coeffs'][('cld', 'planck-atmo')].slope)
data = [ac, fb['fb_coeffs'][('cld','lapse-rate')].slope, fb['fb_coeffs'][('cld', 'water-vapor')].slope, fb['fb_coeffs'][('cld', 'albedo')].slope, fb['fb_coeffs'][('cld','cloud')].slope]
data1 =[-3.24, -0.22,  1.72,  0.58, 0.29]
err = [(fb['fb_coeffs'][('cld', 'planck-surf')].stderr + fb['fb_coeffs'][('cld', 'planck-atmo')].stderr), fb['fb_coeffs'][('cld','lapse-rate')].stderr, fb['fb_coeffs'][('cld', 'water-vapor')].stderr, fb['fb_coeffs'][('cld', 'albedo')].stderr, fb['fb_coeffs'][('cld','cloud')].stderr]
err1 = [0.05, 0.18, 0.14, 0.09, 0.36]
fbnams1 = ['Planck', 'Lapse-rate', 'Water-vapor', 'Albedo', 'Cloud']

fig = plt.figure(figsize=(7,5))
offset=0.05
plt.errorbar(range(len(fbnams1)), data, yerr=err, marker=".", linestyle= 'None', label='by Spectfbcalc', color='g')
plt.errorbar([x+offset for x in range(len(fbnams1))], data1, yerr=err1, marker=".", linestyle= 'None', label='by Zelinka et al. 2020', color='black')
plt.xticks(range(len(fbnams1)), fbnams1)
plt.legend(loc='upper left')
plt.ylabel('[W $m^{-2}$ $K^{-1}$]')

plt.savefig(cart_out +'OUTPUT_new_wv/feedback_vsZelinka(HUANG).pdf')

In [13]:
#CONFRONTO INTERANNUAL
ac = (fb['fb_coeffs'][('cld', 'planck-surf')].slope + fb['fb_coeffs'][('cld', 'planck-atmo')].slope)
data = [ac, fb['fb_coeffs'][('cld','lapse-rate')].slope, fb['fb_coeffs'][('cld', 'water-vapor')].slope, fb['fb_coeffs'][('cld', 'albedo')].slope, fb['fb_coeffs'][('cld','cloud')].slope]
ac1 = (fb_inter['fb_coeffs'][('cld', 'planck-surf')].slope + fb_inter['fb_coeffs'][('cld', 'planck-atmo')].slope)
data1 =[ac1, fb_inter['fb_coeffs'][('cld','lapse-rate')].slope, fb_inter['fb_coeffs'][('cld', 'water-vapor')].slope, fb_inter['fb_coeffs'][('cld', 'albedo')].slope, fb_inter['fb_coeffs'][('cld','cloud')].slope]
err = [(fb['fb_coeffs'][('cld', 'planck-surf')].stderr + fb['fb_coeffs'][('cld', 'planck-atmo')].stderr), fb['fb_coeffs'][('cld','lapse-rate')].stderr, fb['fb_coeffs'][('cld', 'water-vapor')].stderr, fb['fb_coeffs'][('cld', 'albedo')].stderr, fb['fb_coeffs'][('cld','cloud')].stderr]
err1 = [(fb_inter['fb_coeffs'][('cld', 'planck-surf')].stderr + fb_inter['fb_coeffs'][('cld', 'planck-atmo')].stderr), fb_inter['fb_coeffs'][('cld','lapse-rate')].stderr, fb_inter['fb_coeffs'][('cld', 'water-vapor')].stderr, fb_inter['fb_coeffs'][('cld', 'albedo')].stderr, fb_inter['fb_coeffs'][('cld','cloud')].stderr]
fbnams1 = ['planck', 'lapse-rate', 'water-vapor', 'albedo', 'cloud']

fig = plt.figure(figsize=(7,5))
offset=0.05
plt.errorbar(range(len(fbnams1)), data, yerr=err, marker=".", linestyle= 'None', label='total feedbacks', color='g')
plt.errorbar([x+offset for x in range(len(fbnams1))], data1, yerr=err1, marker=".", linestyle= 'None', label='interannual feedbacks', color='black')
plt.xticks(range(len(fbnams1)), fbnams1)
plt.legend(loc='upper left')
plt.ylabel('[W $m^{-2}$ $K^{-1}$]')

plt.savefig(cart_out +'OUTPUT_new_wv/feedback_tot&interannual.pdf')

In [19]:
#CONFRONTO INTERANNUAL DIAGONALE
fbnams1 = ['planck', 'lapse-rate', 'water-vapor', 'albedo', 'cloud']
fig = plt.figure(figsize=(7,5))
plt.scatter(data1, data)
plt.xlabel('interannual feedbacks')
plt.ylabel('feedbacks')
for nome, x, y in zip(fbnams1, data1, data):
    plt.annotate(nome, (x, y))

minimo = min(
    min(data1),
    min(data)
)

massimo = max(
    max(data1),
    max(data)
)

plt.plot(
    [minimo, massimo],
    [minimo, massimo],
    linestyle="--"
)

plt.savefig(cart_out +'OUTPUT_new_wv/feedback(tot vs interannual).pdf')

In [11]:
# ---------- plotting ----------
#val=vals_anom[:150]
sky='cld'
title='dRt-net TOA'
output_file='/perm/ecme5989/new_func_prove/model_wNET/remapped_HUANG/OUTPUT_new_wv/dRt-netTOAHUANG-nomask_cloud.pdf'#path and name output
plt.figure(figsize=(12,5))
plt.plot( NET_cld, color="black", linestyle="-", label="Net TOA Anomaly")

plt.plot(dRt_sum_cloud.values+(NET_cld.values[0]-dRt_sum_cloud.values[0]), color="red", linestyle="-", linewidth=2,
             label="Sum of dRt Components")

plt.ylim(0, 7.5)
plt.xlabel("Year")
plt.ylabel("W/m²")
plt.title("All sky", fontsize=14)
plt.axhline(0, color="gray", linestyle="--", linewidth=1)
plt.legend(loc="upper right")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()

plt.savefig(output_file, dpi=300, bbox_inches="tight")
   

In [10]:
# ---------- plotting ----------
#val=vals_anom[:150]
sky='clr'
title='dRt-net TOA'
output_file='/perm/ecme5989/new_func_prove/model_wNET/remapped_HUANG/OUTPUT_new_wv/dRt-netTOAHUANG-nomask_clear.pdf'#path and name output
plt.figure(figsize=(12,5))
plt.plot( NET_clr, color="black", linestyle="-", label="Net TOA Anomaly")

plt.plot(dRt_sum.values+(NET_clr.values[0]-dRt_sum.values[0]), color="red", linestyle="-", linewidth=2,
             label="Sum of dRt Components")

plt.ylim(0, 7.5)
plt.xlabel("Year")
plt.ylabel("W/m²")
plt.title("Clear sky", fontsize=14)
plt.axhline(0, color="gray", linestyle="--", linewidth=1)
plt.legend(loc="upper right")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()

plt.savefig(output_file, dpi=300, bbox_inches="tight")
   

In [24]:
sky='clr'
comp_labels    = ["albedo", "lapse-rate", "planck-atmo", "planck-surf", "water-vapor"]
color_cycle = ["tab:blue", "tab:green", "tab:purple", "tab:orange", "tab:cyan", "tab:pink", "tab:brown"]

output_file='/perm/ecme5989/new_func_prove/model_wNET/remapped_HUANG/OUTPUT_new_wv/dRt-individuals-nomask_clear.pdf'#path and name output
plt.figure(figsize=(12,5))


for i, lab in enumerate(comp_labels):

    x = np.round(dRt[(sky, lab)].values).astype(int)
    plt.plot(
        x,
        label=lab,
        color=color_cycle[i % len(color_cycle)]
    )

plt.xlabel("Year")
plt.ylabel("W/m²")
plt.title("Clear sky", fontsize=14)
plt.axhline(0, color="gray", linestyle="--", linewidth=1)
plt.legend(loc="upper right")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()

plt.savefig(output_file, dpi=300, bbox_inches="tight")


In [23]:
sky='cld'
comp_labels    = ["albedo", "lapse-rate", "planck-atmo", "planck-surf", "water-vapor", "cloud"]
color_cycle = ["tab:blue", "tab:green", "tab:purple", "tab:orange", "tab:cyan", "tab:pink", "tab:brown"]

output_file='/perm/ecme5989/new_func_prove/model_wNET/remapped_HUANG/OUTPUT_new_wv/dRt-individuals-nomask_allsky.pdf'#path and name output
plt.figure(figsize=(12,5))


for i, lab in enumerate(comp_labels):

    x = np.round(dRt[(sky, lab)].values).astype(int)
    plt.plot(
        x,
        label=lab,
        color=color_cycle[i % len(color_cycle)]
    )

plt.xlabel("Year")
plt.ylabel("W/m²")
plt.title("All sky", fontsize=14)
plt.axhline(0, color="gray", linestyle="--", linewidth=1)
plt.legend(loc="upper right")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()

plt.savefig(output_file, dpi=300, bbox_inches="tight")

In [9]:
#MULTIPLOT RADIATIVE ANOMALIES
fig, axs=plt.subplots(2,2, figsize=(20, 14))
fig.suptitle('Radiative Anomalies', fontsize=28)

color_cycle = ["tab:blue", "tab:green", "tab:purple", "tab:orange", "tab:cyan", "tab:pink", "tab:brown"]
configs = [{"sky": "clr", "ax_top": axs[0, 0],"ax_bottom": axs[1, 0], "net": NET_clr, "sum": dRt_sum, "title": "Clear sky",
        "components": ["albedo", "lapse-rate", "planck-atmo", "planck-surf", "water-vapor"]},
    {"sky": "cld","ax_top": axs[0, 1],"ax_bottom": axs[1, 1],  "net": NET_cld, "sum": dRt_sum_cloud, "title": "All sky",
        "components": ["albedo", "lapse-rate", "planck-atmo", "planck-surf", "water-vapor", "cloud"]    }]

for cfg in configs:

    # --- pannello superiore
    offset = cfg["net"].values[0] - cfg["sum"].values[0]
    cfg["ax_top"].plot( cfg["net"], color="black", lw=2,
        label="Net TOA anomaly")

    cfg["ax_top"].plot(cfg["sum"].values + offset, color="red", lw=2,
        label="Sum of dRt components")

    cfg["ax_top"].set_title(cfg["title"], fontsize=16)
    cfg["ax_top"].set_ylim(0, 7.5)
    cfg["ax_top"].axhline(0, color="gray", ls="--")
    cfg["ax_top"].grid(True, ls="--", alpha=0.5)
    cfg["ax_top"].legend()

    # --- pannello inferiore
    for i, lab in enumerate(cfg["components"]):

        x = dRt[(cfg["sky"], lab)].values   # niente round/int
        cfg["ax_bottom"].plot(x, label=lab, color=color_cycle[i])

    cfg["ax_bottom"].set_title(f"{cfg['title']} components", fontsize=16)
    cfg["ax_bottom"].axhline(0, color="gray",ls="--")
    cfg["ax_bottom"].grid(True,  ls="--", alpha=0.5)
    cfg["ax_bottom"].legend()


for ax in axs.flat:
    ax.set(xlabel='year', ylabel='W/m²')
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig('/perm/ecme5989/new_func_prove/model_wNET/remapped_HUANG/OUTPUT_new_wv/multiplotdR.pdf', dpi=300,
    bbox_inches="tight")

In [ ]:
config_file = '/perm/ecme5989/SpectFbCalc/spectfbcalc/config_r8.yaml'
control, experiment, kernel = sfc.preprocess_data(config_file)

In [22]:
def _guess_names(da):
        # best-effort name guessing for coordinates/dims
        lat = next((n for n in ["lat", "latitude", "nav_lat", "y"] if n in da.coords or n in da.dims), None)
        lon = next((n for n in ["lon", "longitude", "nav_lon", "x"] if n in da.coords or n in da.dims), None)
        time = next((n for n in ["year", "time", "time_counter"] if n in da.coords or n in da.dims), None)
        return lat, lon, time

def _global_mean(da):
        """Area-weighted global mean over spatial dims (cos(lat) weights)."""
        lat, lon, time = _guess_names(da)
        # If we have lat/lon, do weighted mean
        if lat is not None and lon is not None:
            # weights need to be subset of da dims; 1D weights over lat are fine
            weights = np.cos(np.deg2rad(xr.where(np.isfinite(da[lat]), da[lat], 0.0)))
            gm = da.weighted(weights).mean(dim=[lat, lon])
        else:
            # Fallback: mean over all non-time dims
            spatial_dims = [d for d in da.dims if d not in {"time", "time_counter", "year"}]
            gm = da.mean(dim=spatial_dims)
        return gm

def _get_time(da):
        lat, lon, time = _guess_names(da)
        if time is None:
            # fallback: take first dim as "time-like"
            time = da.dims[0]
        return time

    # ---------- load sim/control & compute anomaly ----------

def _global_mean(da):
        """Area-weighted global mean over spatial dims (cos(lat) weights)."""
        lat, lon, time = _guess_names(da)
        # If we have lat/lon, do weighted mean
        if lat is not None and lon is not None:
            # weights need to be subset of da dims; 1D weights over lat are fine
            weights = np.cos(np.deg2rad(xr.where(np.isfinite(da[lat]), da[lat], 0.0)))
            gm = da.weighted(weights).mean(dim=[lat, lon])
        else:
            # Fallback: mean over all non-time dims
            spatial_dims = [d for d in da.dims if d not in {"time", "time_counter", "year"}]
            gm = da.mean(dim=spatial_dims)
        return gm

def somma(dRt, sky='clr'):
  
    if sky == 'clr':
        # prendi solo le componenti clear-sky
        arrays = [v for (s, comp), v in dRt.items() if s == 'clr']
    else:
        # prendi tutte le componenti all-sky + cloud
        arrays = [
            v for (s, comp), v in dRt.items()
            if s != 'clr' or comp == 'cloud'
        ]

    # somma tutte le DataArray
    dRt_sum = sum(arrays[1:], arrays[0])
    
    return dRt_sum

cart='/perm/ecme5989/new_func_prove/model_wNET/remapped_HUANG/no_mask/'
dRt=sfc.open_dRt(cart)
default_labels = ["Albedo", "Lapse Rate", "Planck Atmos", "Planck Surface", "Water Vapor", 'Cloud']
dRt_sum_cloud = somma(dRt, sky='cld')
dRt_sum_nocloud = somma(dRt, sky='clr')


tnr_anomaly= experiment.ds_anom['Net0'].groupby("time.year").mean(dim='time')
NET_clr  = _global_mean(tnr_anomaly)
years_anom      = NET_clr["year"].values
vals_anom       = NET_clr.values

tnr_anomaly= experiment.ds_anom['Net'].groupby("time.year").mean(dim='time')
NET_cld  = _global_mean(tnr_anomaly)

In [25]:
#calcl fb
#gtas
num=10
gtas = ctl.global_mean(experiment.ds_anom['tas']).groupby('time.year').mean('time')
start_year = int(gtas.year.min()) 
gtas1 = gtas.groupby((gtas.year-start_year) // num * num).mean()

start_year = int(dRt_sum_cloud.year.min())
dRt_cld=dRt_sum_cloud.groupby((dRt_sum_cloud.year-start_year) // num * num).mean()
start_year = int(dRt_sum_nocloud.year.min())
dRt_clr=dRt_sum_nocloud.groupby((dRt_sum_nocloud.year-start_year) // num * num).mean()
start_year = int(NET_clr.year.min())
N_clr=NET_clr.groupby((NET_clr.year-start_year) // num * num).mean()
start_year = int(NET_cld.year.min())
N_cld=NET_cld.groupby((NET_cld.year-start_year) // num * num).mean()


fb_sum_cld= stats.linregress(gtas1, dRt_cld)
fb_sum_clr= stats.linregress(gtas1, dRt_clr)
fb_cld= stats.linregress(gtas1, N_cld)
fb_clr= stats.linregress(gtas1, N_clr)

In [27]:
fb_cld

LinregressResult(slope=np.float64(-0.812329090878592), intercept=np.float64(6.784153083557609), rvalue=np.float64(-0.9923420290359717), pvalue=np.float64(3.40502450956213e-13), stderr=np.float64(0.028043857430692112), intercept_stderr=np.float64(0.17490163026514902))

In [28]:
#GREGORY PLOT
output_file='/perm/ecme5989/new_func_prove/model_wNET/remapped_HUANG/OUTPUT/Gregory_clr.pdf'#path and name output
plt.figure(figsize=(12,5))

plt.scatter(gtas, dRt_sum_nocloud , color='blue', label='dRt_sum, feedback slope:-1.127')
plt.scatter(gtas, NET_clr , color='green', label='Net_TOA, feedback slope: -0.8146')
# Linea di regressione
x_fit = np.linspace(gtas.values.min(), gtas.values.max(), 100)
y_fit = fb_sum_clr.slope * x_fit + fb_sum_clr.intercept
plt.plot(x_fit, y_fit, color='blue')

y_fit1 = fb_clr.slope * x_fit + fb_clr.intercept
plt.plot(x_fit, y_fit1, color='green')

plt.xlabel("gtas")
plt.ylabel("W/m²")
plt.legend()
plt.tight_layout()
plt.savefig(output_file)

In [29]:
#GREGORY PLOT
output_file='/perm/ecme5989/new_func_prove/model_wNET/remapped_HUANG/OUTPUT/Gregory_cld.pdf'#path and name output
plt.figure(figsize=(12,5))

plt.scatter(gtas, dRt_sum_cloud , color='blue', label='dRt_sum, feedback slope:-1.125')
plt.scatter(gtas, NET_cld , color='green', label='Net_TOA, feedback slope: -0.8123')
# Linea di regressione
x_fit = np.linspace(gtas.values.min(), gtas.values.max(), 100)
y_fit = fb_sum_cld.slope * x_fit + fb_sum_cld.intercept
plt.plot(x_fit, y_fit, color='blue')

y_fit1 = fb_cld.slope * x_fit + fb_cld.intercept
plt.plot(x_fit, y_fit1, color='green')

plt.xlabel("gtas")
plt.ylabel("W/m²")
plt.legend()
plt.tight_layout()
plt.savefig(output_file)